# Using Ibis to Work With Database

**Author:** Shinin Varongchayakul

**Date:** 25 Jul 2026

## 1. Getting Started

In [ ]:
# Install the package
! pip install ibis-framework

In [1]:
# Import the package
import ibis

## 2. Connecting the Database

In [2]:
# Connect to the database
con = ibis.sqlite.connect("chinook.sqlite")

In [ ]:
# List the tables
con.list_tables()

['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

## 3. Loading a Table

In [7]:
# Load a table
tracks = con.table("Track")

In [ ]:
# View the table schema
tracks.schema()

ibis.Schema {
  TrackId       !int64
  Name          !string(200)
  AlbumId       int64
  MediaTypeId   !int64
  GenreId       int64
  Composer      string(220)
  Milliseconds  !int64
  Bytes         int64
  UnitPrice     !decimal(10, 2)
}

In [9]:
# Preview the data
customers.head().execute()

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,NaN,342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99
3,4,Restless and Wild,3,2,1,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",252051,4331779,0.99
4,5,Princess of the Dawn,3,2,1,Deaffy & R.A. Smith-Diesel,375418,6290521,0.99


## 4. Selecting

In [ ]:
# Select data from the table
(
    tracks.select(
        "Name",
        "GenreId",
        "Milliseconds",
        "UnitPrice"
    )
    .execute()
)

,Name,GenreId,Milliseconds,UnitPrice
0,For Those About To Rock (We Salute You),1,343719,0.99
1,Balls to the Wall,1,342562,0.99
2,Fast As a Shark,1,230619,0.99
3,Restless and Wild,1,252051,0.99
4,Princess of the Dawn,1,375418,0.99
...,...,...,...,...
3498,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...,24,286741,0.99
3499,"String Quartet No. 12 in C Minor, D. 703 ""Quar...",24,139200,0.99
3500,"L'orfeo, Act 3, Sinfonia (Orchestra)",24,66639,0.99
3501,"Quintet for Horn, Violin, 2 Violas, and Cello ...",24,221331,0.99


## 5. Sorting

In [39]:
# Sort the data
(
    tracks.order_by(
        tracks.Milliseconds.desc() # Sort by duration in descending order
    )
    .select(
        "Name",
        "GenreId",
        "Milliseconds",
        "UnitPrice"
    )
    .execute()
)

,Name,GenreId,Milliseconds,UnitPrice
0,Occupation / Precipice,19,5286953,1.99
1,Through a Looking Glass,21,5088838,1.99
2,"Greetings from Earth, Pt. 1",20,2960293,1.99
3,The Man With Nine Lives,20,2956998,1.99
4,"Battlestar Galactica, Pt. 2",20,2956081,1.99
...,...,...,...,...
3498,Commercial 1,17,7941,0.99
3499,Oprah,4,6635,0.99
3500,A Statistic,4,6373,0.99
3501,Now Sports,4,4884,0.99


## 6. Filtering

In [ ]:
# Filter the data
(
    tracks.filter(
        tracks.GenreId == 1 # Select rock songs
    )
    .select(
        "Name",
        "GenreId",
        "Milliseconds",
        "UnitPrice"
    )
    .execute()
)

,Name,GenreId,Milliseconds,UnitPrice
0,For Those About To Rock (We Salute You),1,343719,0.99
1,Balls to the Wall,1,342562,0.99
2,Fast As a Shark,1,230619,0.99
3,Restless and Wild,1,252051,0.99
4,Princess of the Dawn,1,375418,0.99
...,...,...,...,...
1292,Tease Me Please Me,1,287229,0.99
1293,Wind of Change,1,315325,0.99
1294,Send Me an Angel,1,273041,0.99
1295,I Guess You're Right,1,212044,0.99


## 7. Joining

In [30]:
# Get the genres table
genres = con.table("Genre")

# Join the tables
joined = (
    tracks.join(
        genres,
        tracks.GenreId == genres.GenreId
    ).select(
        TrackName=tracks.Name,
        Genre=genres.Name,
        Duration=tracks.Milliseconds,
        UnitPrice=tracks.UnitPrice

    )
)

# View the new table
joined.execute()

,TrackName,Genre,Duration,UnitPrice
0,For Those About To Rock (We Salute You),Rock,343719,0.99
1,Balls to the Wall,Rock,342562,0.99
2,Fast As a Shark,Rock,230619,0.99
3,Restless and Wild,Rock,252051,0.99
4,Princess of the Dawn,Rock,375418,0.99
...,...,...,...,...
3498,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...,Classical,286741,0.99
3499,"String Quartet No. 12 in C Minor, D. 703 ""Quar...",Classical,139200,0.99
3500,"L'orfeo, Act 3, Sinfonia (Orchestra)",Classical,66639,0.99
3501,"Quintet for Horn, Violin, 2 Violas, and Cello ...",Classical,221331,0.99


## 8. Aggregating

In [36]:
# Aggregate the data
(
    joined.group_by(
        joined.Genre
    )
    .aggregate(
        TrackCount=joined.count()
    )
    .order_by(
        ibis.desc("TrackCount")
    )
    .execute()
)

,Genre,TrackCount
0,Rock,1297
1,Latin,579
2,Metal,374
3,Alternative & Punk,332
4,Jazz,130
5,TV Shows,93
6,Blues,81
7,Classical,74
8,Drama,64
9,R&B/Soul,61


## 8. Using SQL

In [ ]:
# Using SQL in Ibis
(
    con.sql(
        """
        SELECT
            Name,
            GenreId,
            Milliseconds,
            UnitPrice
        FROM
            Track
        WHERE
            GenreId = 1;
        """
    )
    .execute()
)

,Name,GenreId,Milliseconds,UnitPrice
0,For Those About To Rock (We Salute You),1,343719,0.99
1,Balls to the Wall,1,342562,0.99
2,Fast As a Shark,1,230619,0.99
3,Restless and Wild,1,252051,0.99
4,Princess of the Dawn,1,375418,0.99
...,...,...,...,...
1292,Tease Me Please Me,1,287229,0.99
1293,Wind of Change,1,315325,0.99
1294,Send Me an Angel,1,273041,0.99
1295,I Guess You're Right,1,212044,0.99
